In [1]:
import sympy as sp
import scipy as sc
import numpy as np
from time import sleep,time

In [2]:
F = lambda x : x * sp.exp(-x)
G = lambda x,omega : x + omega * F(x)

2.2.b. 

In [ ]:
t = sp.symbols('t')
dF_dt = sp.diff(F(t),t)

x0 = 5
omega = -1/dF_dt.subs({t:x0})
eps = 1.e-6

## test |F(x)|<eps
x=x0
cpt=0
while abs(F(x)) > eps:
    x = G(x,omega)
    cpt+=1
print(f" x={x},\n erreur={abs(x-np.sqrt(2))},\n iterations={cpt}")

# test |x_courant - x_maj|<eps
x_courant=x0
cpt=0
d=1
while abs(d) > eps:
    x_maj = G(x_courant,omega)
    d=x_courant-x_maj
    x_courant = x_maj
    cpt+=1
print(f" x={x},\n erreur={abs(x-np.sqrt(2))},\n iterations={cpt}")

def pt_fixe(omega,x0,eps):
    x=x0
    cpt=0
    while abs(F(x)) > eps:
        x = G(x,omega)
        cpt+=1
    return [x,cpt]

precision=[10**(-k) for k in range(2,15)]
iterations=[]

for eps in precision:
    iterations.append(pt_fixe(omega,x0,eps)[1])

from matplotlib.pyplot import semilogx

semilogx(precision,iterations)

In [7]:
def F(X):
    a,b = X
    t1 = a + sp.sqrt(abs(a)+2)+b - sp.sqrt(2)
    t2 = a**2 + sp.cos(b) -1 
    return sp.Matrix([[t1,t2]]).T

print('F(0,0)=',F(sp.Matrix([0., 0.])))
print('F(1/2 , 1/2)=',F(sp.Matrix([0.5, 0.5])))

F(0,0)= Matrix([[0], [0]])
F(1/2 , 1/2)= Matrix([[2.58113883008419 - sqrt(2)], [0.127582561890373]])


3.1. Jacobienne de f

In [8]:
x,y = sp.symbols('x y', real=True)
dF_dx = sp.diff(F([x,y]),x)
dF_dy = sp.diff(F([x,y]),y)

JF = sp.Matrix([dF_dx.T,dF_dy.T])
JF

Matrix([
[1 + sign(x)/(2*sqrt(Abs(x) + 2)),     2*x],
[                               1, -sin(y)]])

3.2. Methode de point fixe 

In [13]:
# JF.subs({x:0.,y:0.}).inv()
JF.subs({x:0.5,y:0.5}).inv()

display(JF.subs({x:0.5,y:0.5}).inv())


jf = sp.lambdify((x,y),JF, modules='numpy')
display(jf(0.,0.))
display(jf(0.5,0.5))

Matrix([
[0.293939778138118,   0.61310830247778],
[ 0.61310830247778, -0.806990171296704]])

array([[ 1.,  0.],
       [ 1., -0.]])

array([[ 1.31622777,  1.        ],
       [ 1.        , -0.47942554]])

3.3. Newton

In [14]:
J=JF.subs({x:0.5,y:0.5})
OMEGA=J.inv()

G = lambda xx,omega : xx + omega * F(xx)

xx0 = sp.Matrix([0.5,0.5])
omega = -JF.subs({x:xx0[0],y:xx0[1]}).inv()
eps = 1.e-14

## test |F(x)|<eps
xx=xx0
cpt=0
while F(xx).norm() > eps:
    yy = JF.subs({x:xx[0],y:xx[1]}).LUsolve(F(xx))
    xx = xx-yy
    cpt+=1
print(f" x={xx},\n iterations={cpt}")
display(xx)


Matrix([
[0.293939778138118,   0.61310830247778],
[ 0.61310830247778, -0.806990171296704]])

KeyboardInterrupt: 